# Phase 6: Hierarchical Demand Forecasting & Optimal Reconciliation
## MarketPulse: E-Commerce Demand Forecasting Engine

This research notebook investigates **hierarchical time-series demand forecasting** across the 12 levels of the M5 Walmart hierarchy.

### Core Concepts:
1. **Retail Hierarchy**: Corporate Total $\to$ State $\to$ Store $\to$ Category $\to$ Department $\to$ SKU (Store-Item).
2. **Summing Matrix ($S$)**: Binary matrix mapping bottom nodes to all aggregate nodes: $\boldsymbol{y}_t = S \boldsymbol{y}_{b,t}$.
3. **Incoherence Problem**: Independent forecasters at different aggregation levels inevitably contradict each other (e.g., $\sum \hat{y}_{\text{store}} \neq \hat{y}_{\text{state}}$).
4. **Optimal Reconciliation**: Solving $\tilde{\boldsymbol{y}} = S P \hat{\boldsymbol{y}}$ via:
   - Bottom-Up (BU)
   - Top-Down (TD-PHA)
   - Ordinary Least Squares (OLS): $P_{\text{OLS}} = (S^T S)^{-1} S^T$
   - Minimum Trace (MinT) with Ledoit-Wolf Covariance Shrinkage
   - Non-Negative NNLS Reconciliation (enforcing $\tilde{\boldsymbol{y}} \ge 0$)

In [ ]:
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.hierarchy import M5Hierarchy
from src.hierarchy.hierarchy_builder import HierarchyTree
from src.hierarchy.aggregation import HierarchicalAggregator
from src.hierarchy.reconciliation import HierarchicalReconciler

## 1. Load Processed Dataset & Construct Hierarchy Tree

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'processed' / 'm5_processed.parquet'
df = pd.read_parquet(data_path)
print(f"Loaded processed M5 dataset: {df.shape[0]:,} records across {df['store_id'].nunique()} stores and {df['item_id'].nunique()} items.")

tree = HierarchyTree(metadata_df=df)
summary_df = tree.get_level_summary()
summary_df

## 2. Inspecting the Hierarchical Summing Matrix $S$

In [ ]:
S, total_names, bottom_names = tree.build_summing_matrix()
print(f"Summing Matrix S Shape: {S.shape[0]} total nodes x {S.shape[1]} bottom SKU series")
print(f"Matrix Sparsity: {(1.0 - np.count_nonzero(S) / S.size) * 100:.2f}%")

# Verification: every bottom series maps to exactly 12 hierarchy levels
col_sums = S.sum(axis=0)
assert np.all(col_sums == 12.0), "Summing matrix check failed: column sum != 12"
print("Summing matrix verified: Strict hierarchy aggregation holds across all 12 levels.")

## 3. Coherence Audit on Independent Base Forecasts

In [ ]:
from scripts.run_reconciliation import generate_base_forecasts

aggregator = HierarchicalAggregator(tree.m5_hierarchy)
bottom_wide = aggregator.prepare_bottom_wide_dataframe(df)
all_wide = aggregator.aggregate_bottom_to_all_levels(bottom_wide)

horizon = 28
train_size = len(all_wide) - horizon
y_train_all = all_wide.iloc[:train_size]
y_test_all = all_wide.iloc[train_size:]

# Generate independent base forecasts across all nodes
in_sample_base, base_forecasts = generate_base_forecasts(
    y_train_wide=y_train_all,
    hierarchy=tree.m5_hierarchy,
    horizon=horizon,
    sp=7
)

# Verify incoherence
incoh_base = aggregator.verify_coherence(base_forecasts)
print("Base Independent Forecast Coherence:")
print(f" - Is Coherent:           {incoh_base['is_coherent']}")
print(f" - Max Incoherence:       {incoh_base['max_abs_incoherence']:.4f} units")
print(f" - Relative Incoherence:  {incoh_base['relative_incoherence'] * 100:.2f}%")

## 4. Optimal Reconciliation (MinT-Shrink & Non-Negative NNLS)

In [ ]:
# Reconcile via Non-Negative MinT
reconciler = HierarchicalReconciler(
    hierarchy=tree.m5_hierarchy,
    method='mint_shrink',
    non_negative=True,
)
reconciler.fit(
    in_sample_actuals=y_train_all,
    in_sample_base_forecasts=in_sample_base,
)
reconciled_forecasts = reconciler.reconcile(base_forecasts)

# Verify coherence post-reconciliation
incoh_rec = aggregator.verify_coherence(reconciled_forecasts)
print("Reconciled Forecast Coherence:")
print(f" - Is Coherent:           {incoh_rec['is_coherent']}")
print(f" - Max Incoherence:       {incoh_rec['max_abs_incoherence']:.6f} units")
print(f" - Non-negativity satisfied: {(reconciled_forecasts.values >= 0).all()}")

## 5. Performance Impact Across Hierarchy Levels

In [ ]:
impact_df = HierarchicalReconciler.evaluate_reconciliation_impact(
    base_forecasts=base_forecasts,
    reconciled_forecasts=reconciled_forecasts,
    actuals=y_test_all,
    hierarchy=tree.m5_hierarchy,
)
impact_df[['level', 'level_name', 'base_wape', 'rec_wape', 'wape_improvement_pct', 'base_rmse', 'rec_rmse']]

## 6. Visualization: Coherent Total Corporate Forecast vs Actuals

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5), dpi=150)
ax.plot(y_test_all.index, y_test_all['Total'], label='Ground Truth Actuals', color='black', lw=2.0)
ax.plot(base_forecasts.index, base_forecasts['Total'], label='Base Forecast (Incoherent)', color='crimson', lw=1.8, linestyle='--')
ax.plot(reconciled_forecasts.index, reconciled_forecasts['Total'], label='Reconciled Forecast (NN-MinT, Coherent)', color='navy', lw=2.0)
ax.set_title('Level 1: Corporate Total Demand - Before vs After Reconciliation', fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales Units')
ax.grid(True, alpha=0.3)
ax.legend(frameon=True)
plt.tight_layout()
plt.show()